# TileGuard — Signed Offset Distribution Visualization

This notebook provides interactive visualizations of the coordinate offset distribution analysis (Step 1.2). It analyzes 148,268 out-of-range coordinates across 294 production vector tiles (from OpenMapTiles, OpenFreeMap, and CARTO Streets) to classify diagnostic patterns.

In [ ]:
# Install dependencies if they are not already installed
!pip install pandas plotly nbformat

In [ ]:
import json
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Set dark template for premium design aesthetics
plotly_template = "plotly_dark"

# Load the distribution dataset
with open('../analysis/offset-distribution.json', 'r') as f:
    data = json.load(f)

summary = data['summary']
print(f"Total Out-of-Range Coordinates: {summary['totalEntries']:,}")
print(f"Per-Dataset Breakdown: {summary['perDataset']}")
print(f"Direction Breakdown: {summary['directionBreakdown']}")

## 1. Signed Offset Histogram

Visualizes the bimodal distribution of offsets. The two main peaks at `±1 to ±64` (geometry clipping buffers) and `±1025 to ±4096` (label duplication centroids) are clearly visible.

In [ ]:
hist_data = data['histogram']
# Sort buckets logically (negatives to positives)
def bucket_sort_key(item):
    k = item[0]
    if k == '0 (in-range)': return 0
    import re
    match = re.match(r'([+-])(\d+|>?\d+)', k)
    if not match: return 0
    sign = -1 if match.group(1) == '-' else 1
    val = 99999 if match.group(2).startswith('>') else int(match.group(2))
    return sign * val

sorted_hist = sorted(hist_data.items(), key=bucket_sort_key)
buckets = [item[0] for item in sorted_hist]
counts = [item[1] for item in sorted_hist]
colors = ['#ef4444' if b.startswith('-') else '#06b6d4' for b in buckets]

fig_hist = go.Figure(data=[go.Bar(
    x=buckets,
    y=counts,
    marker_color=colors,
    opacity=0.85,
    hovertemplate='<b>%{x}</b><br>Count: %{y:,}<extra></extra>'
)])

fig_hist.update_layout(
    title="Signed Offset Histogram (Bimodal Distribution)",
    template=plotly_template,
    xaxis_title="Offset Bucket (units beyond extent)",
    yaxis_title="Coordinate Count",
    height=500,
    bargap=0.15
)
fig_hist.show()

## 2. Layer & Geometry Breakdown

This breakdown highlights which layers and geometry types produce the out-of-range coordinates. Notice that `place` and `countries` account for the vast majority of coordinates.

In [ ]:
# Create subplots: Layer Bar Chart + Geometry Pie Chart
fig_breakdown = make_subplots(rows=1, cols=2, specs=[[{'type': 'bar'}, {'type': 'pie'}]], subplot_titles=("Layer Breakdown", "Geometry Types"))

# 1. Layer chart
layers_sorted = sorted(data['layerBreakdown'].items(), key=lambda x: x[1])
fig_breakdown.add_trace(
    go.Bar(
        y=[x[0] for x in layers_sorted],
        x=[x[1] for x in layers_sorted],
        orientation='h',
        marker_color='#6366f1',
        showlegend=False,
        hovertemplate='<b>%{y}</b><br>Count: %{x:,}<extra></extra>'
    ),
    row=1, col=1
)

# 2. Geometry Pie chart
geom_data = data['geometryTypeBreakdown']
fig_breakdown.add_trace(
    go.Pie(
        labels=list(geom_data.keys()),
        values=list(geom_data.values()),
        hole=0.4,
        marker=dict(colors=['#6366f1', '#06b6d4', '#f59e0b']),
        textinfo='label+percent',
        hovertemplate='<b>%{label}</b><br>%{value:,} (%{percent})<extra></extra>'
    ),
    row=1, col=2
)

fig_breakdown.update_layout(
    title_text="Layer & Geometry Analysis",
    template=plotly_template,
    height=450
)
fig_breakdown.show()

## 3. Direction Balance per Dataset

Symmetry analysis: comparing `below-zero` vs `above-extent` counts across each provider.

In [ ]:
ds_names = list(summary['perDataset'].keys())
total_vals = list(summary['perDataset'].values())
below_ratio = summary['directionBreakdown']['below-zero'] / summary['totalEntries']
above_ratio = summary['directionBreakdown']['above-extent'] / summary['totalEntries']

fig_dir = go.Figure(data=[
    go.Bar(name='Below Zero', x=ds_names, y=[int(v * below_ratio) for v in total_vals], marker_color='#ef4444', opacity=0.8),
    go.Bar(name='Above Extent', x=ds_names, y=[int(v * above_ratio) for v in total_vals], marker_color='#06b6d4', opacity=0.8)
])

fig_dir.update_layout(
    title="Offset Direction Balance per Dataset",
    template=plotly_template,
    barmode='stack',
    yaxis_title="Coordinate Count",
    height=400
)
fig_dir.show()

## 4. Cross-Provider Identity Verification

Verifying that coordinates are not coincidences by looking at features that match exactly by name and layer across providers.

In [ ]:
cp = data['crossProviderIdentity']
labels = ['Confirmed (Same Name/Class)', 'Unconfirmed (No Shared Name)']
values = [cp['confirmedSameFeature'], cp['unconfirmedSameFeature']]

fig_cp = go.Figure(data=[go.Pie(
    labels=labels,
    values=values,
    hole=0.45,
    marker=dict(colors=['#22c55e', '#f59e0b']),
    textinfo='label+value+percent',
    hovertemplate='<b>%{label}</b><br>%{value:,} (%{percent})<extra></extra>'
)])

fig_cp.update_layout(
    title="Cross-Provider Feature Identity Match (Property Verification)",
    template=plotly_template,
    height=450,
    showlegend=True
)
fig_cp.show()

## 5. Geographic Coordinate Scatter (Spatial Distribution)

Plots the out-of-range coordinates in coordinate space for the global z=0 tile `0-0-0` to see their spatial distribution relative to the 0-4096 extent box.

In [ ]:
# Filter entries for global tile 0-0-0
df_entries = pd.DataFrame(data['entries'])
df_tile = df_entries[df_entries['tile'] == '0-0-0']

fig_spatial = px.scatter(
    df_tile,
    x="x",
    y="y",
    color="layer",
    symbol="geometryType",
    title="Out-of-Range Coordinates Spatial Distribution (Tile 0-0-0)",
    labels={"x": "X Coordinate", "y": "Y Coordinate"},
    hover_data=["dataset", "xOffset", "yOffset"]
)

# Draw the tile boundary box (0,0 to 4096,4096)
fig_spatial.add_shape(
    type="rect",
    x0=0, y0=0, x1=4096, y1=4096,
    line=dict(color="white", width=2, dash="dash"),
    fillcolor="rgba(255, 255, 255, 0.05)"
)

fig_spatial.update_layout(
    template=plotly_template,
    height=600,
    width=700,
    xaxis=dict(range=[-4500, 8500]),
    yaxis=dict(range=[-4500, 8500]),
    legend=dict(yanchor="top", y=0.99, xanchor="left", x=1.05)
)
fig_spatial.show()